# ZTE on Google Colab — end to end

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victor-iyi/zte/blob/main/notebooks/zte_colab.ipynb)

**Download → prepare → train → benchmark → visualise → pack → download.** Train on a powerful Colab Pro GPU, then zip the finished experiments and download them to run **inference locally on your Mac**.

- **Platform-adaptable & auto-accelerated.** `--device auto` (the default) picks **CUDA (Colab GPU) → Cloud TPU (torch_xla) → Apple MPS → CPU**. Nothing to configure.
- **Resumable.** Every long run is `--resume`-safe: if the runtime disconnects, re-run the cell and it continues from the last checkpoint.
- **No Colab surprises.** Section 2 sets the env vars Colab leaves unset and fixes the working directory / output paths so the CLIs never error on a fresh runtime.

> ZTE requires **Python 3.14** (Colab ships an older Python), so we use [`uv`](https://docs.astral.sh/uv/) to provision it — one cell, no system changes. All ZTE code therefore runs via `!uv run …` (the 3.14 venv), not the notebook's own kernel.

**Pick a GPU runtime now:** `Runtime → Change runtime type → T4/A100 GPU` (or `TPU`).

## 1 · Set up (uv provisions Python 3.14 + installs ZTE)

In [ ]:
import os

!pip install -q uv
if not os.path.isfile('pyproject.toml') and not os.path.isdir('zte'):
    !git clone --depth 1 https://github.com/victor-iyi/zte.git
if os.path.isdir('zte'):
    %cd zte
# Provision Python 3.14 + install torch (CUDA wheel on a Colab GPU) and all extras. Cached across runs.
!uv python install 3.14
!uv sync --group all

## 2 · Bootstrap the environment (fixes the usual Colab errors)
Colab does not set the env vars headless plotting / tokenizers expect, and the CLIs use paths relative to the repo root. This cell sets those vars for every `!uv run …` subprocess and creates the `res/` output directories, so nothing errors later. Idempotent — safe to re-run.

In [ ]:
import os

# Set in the notebook kernel so every `!uv run` subprocess inherits them.
os.environ.setdefault('MPLBACKEND', 'Agg')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('MPLCONFIGDIR', os.path.abspath('res/.cache/matplotlib'))
# Create res/ dirs + report the resolved root/accelerator via the 3.14 venv.
!uv run python -c "from zte.utils import bootstrap; import json; print(json.dumps(bootstrap(chdir=True, quiet=True), default=str, indent=1))"

## 3 · Confirm the accelerator (GPU / TPU / CPU)
On a GPU runtime this reports CUDA; on a TPU runtime with `torch_xla` it reports a Cloud TPU; otherwise CPU.

In [ ]:
!uv run python -c "from zte.utils import accelerator_info; import json; print(json.dumps(accelerator_info(), indent=1))"

### (Optional) Cloud TPU
On a **TPU** runtime, install `torch_xla` so `--device auto` selects it. `torch_xla` must match the installed torch; this is best-effort (GPU is the primary, tested path). Uncomment to try:

In [ ]:
# !uv pip install -q torch_xla   # then re-run the accelerator cell above; it should report a Cloud TPU

## 4 · Get data (mount Google Drive)
**A) Synthetic (default, no dataset).** A fabricated ZuCo tree — validates the whole pipeline in minutes. The `--synthetic` cells below use it automatically; skip the Drive cell if that is all you want.

**B) Real ZuCo from Google Drive.** Mount Drive and read the dataset **directly** — no `zte-download` / `gdown` (mounting is faster and avoids re-downloading). `zte-run` reads the `.mat` files from Drive once during *prepare*, then caches locally, so training I/O stays fast.

In [ ]:
from google.colab import drive  # type: ignore[import-untyped]

drive.mount('/gdrive')

import os

DATA_DIR = '/gdrive/My Drive/Sharables/ZuCo Dataset'  # the ZuCo .mat files on your Drive
DRIVE_DIR = '/gdrive/My Drive/Sharables'  # where run archives are saved back to
print('data found:', os.path.isdir(DATA_DIR))
!ls "{DATA_DIR}" | head

## 5 · Train an experiment
One full run: prepare → train → evaluate → explore, catalogued under `res/experiments/<name>/`. This uses the flagship EEG-only invariance recipe (with the §10 anti-cone / whitening / meaning fixes enabled). Swap `--synthetic` for `--root <folder>` to use real data.

In [ ]:
!uv run zte-run --config experiments/exp6_skipgram_eegonly_invariant.yaml --synthetic --epochs 5 --name colab_exp6
# Real data (reads .mat straight from Drive; keep the quotes — the path has spaces):
# !uv run zte-run --config experiments/exp6_skipgram_eegonly_invariant.yaml --root "{DATA_DIR}" --name exp6

## 6 · The LOSO “new brain” sweep (resumable)
Trains the invariance recipe once per held-out subject, turning one data point into a **trend**, then builds a combined comparison. `SMOKE=1` = fast synthetic dry-run; drop it and pass a data root for the real multi-hour GPU run. Disconnected? Re-run this cell — finished subjects are skipped, the interrupted one resumes.

In [ ]:
!SMOKE=1 bash scripts/run_loso.sh
# Real GPU sweep (data straight from Drive):   !bash scripts/run_loso.sh "{DATA_DIR}"
# Persist runs to Drive as they finish:        !OUT_ROOT="{DRIVE_DIR}/zte/loso" bash scripts/run_loso.sh "{DATA_DIR}"
# Add the no-recipe control arm too:           !CONTROL=1 bash scripts/run_loso.sh "{DATA_DIR}"

## 7 · Benchmark objectives (fixed-seed sweep)

In [ ]:
!uv run zte-benchmark --synthetic --objectives skipgram,masked --pos-encodings rope --eye-tracking off --seeds 42 --epochs 3 --out res/benchmark
import pandas as pd

pd.read_csv('res/benchmark/benchmark.csv')

## 8 · Visualise & interact (HTML)
Build the interactive **Thought-Space Explorer** + **Neuron Atlas** for a run, and the **comparison dashboard** across all runs. The dashboard is small enough to render inline; the 5 MB explorers are best downloaded (Section 9) and opened locally for full 3-D interaction.

In [ ]:
!uv run zte-visualize --run res/experiments/colab_exp6 --kind both
!uv run zte-compare --experiments res/experiments --out res/experiments/COMPARE.html
from IPython.display import HTML  # type: ignore[import-untyped]

HTML(filename='res/experiments/COMPARE.html')  # the scorecard + best-run dashboard, inline

## 9 · Save / download runs (Colab → Drive → your Mac)
Zip the runs small — checkpoints + config + evaluation only (the heavy cache / TensorBoard / dataset bundle are excluded; a checkpoint already embeds everything inference needs). `--best-only` keeps just `best.pt` for the smallest, inference-only archive; `--move` frees local Colab space after the archive is safely on Drive. Point `--out` at your mounted Drive folder to upload directly.

In [ ]:
!uv run zte-pack list

In [ ]:
# Upload a small, inference-only archive straight to Drive (best.pt per run).
!uv run zte-pack zip --all --best-only --out "{DRIVE_DIR}/zte_experiments.zip"

# Optionally also download it to this browser:
# from google.colab import files
# files.download(f"{DRIVE_DIR}/zte_experiments.zip")

# MOVE instead — zip to Drive, then delete local runs to free Colab space:
# !uv run zte-pack zip --all --best-only --move --out "{DRIVE_DIR}/zte_experiments.zip"

# Restore later (a new Colab session, or your Mac):
# !uv run zte-pack unpack "{DRIVE_DIR}/zte_experiments.zip" --dest res/experiments

## 10 · Run it locally on your Mac (inference)
Grab `zte_experiments.zip` from your Drive (`Sharables/`), then in a terminal on your Mac (Apple-silicon MPS is picked up automatically):

```sh
uv sync --group all
uv run zte-pack unpack ~/Downloads/zte_experiments.zip --dest res/experiments   # or straight from a synced Drive path

# Re-open the interactive explorer for a run:
open res/experiments/exp6/evaluation/interactive/thought_space_explorer.html

# Extract embeddings from the trained checkpoint (best.pt is enough — shapes + normaliser are baked in):
uv run zte-extract --ckpt res/experiments/exp6/checkpoints/best.pt --root "/path/to/ZuCo Dataset" --out res/embeddings/exp6.npz

# Or re-run just the evaluation / visualisation locally:
uv run zte-compare --experiments res/experiments
```

Free Colab space when done: `!uv run zte-pack delete colab_exp6 --yes` (or use `--move` when zipping).